## Imports

In [68]:
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import LinearSVR
from sklearn.metrics import (
    mean_absolute_error as mae,
    mean_squared_error as mse,
    r2_score
)
import joblib
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

## Load Data

In [42]:
df = pd.read_csv("../data/train_test.csv")
val_df = pd.read_csv("../data/validation.csv")
dec_df = pd.read_csv("../data/december_chart_inputs.csv")
template = pd.read_csv("../data/validation_predictions_template.csv")

print("Train shape      :", df.shape)
print("Validation shape :", val_df.shape)
print("Template shape   :", template.shape)
print("_"*50)

print("Train date range :", df["date"].min(), "→", df["date"].max())
print("Val date range   :", val_df["date"].min(), "→", val_df["date"].max())
print("_"*50)

print(f"December Shape: {dec_df.shape}")
print(f"    Dates: {dec_df['date'].min()} -> {dec_df['date'].max()}")
print(f"    Columns: {dec_df.columns}")

print("_"*50)
df.head()

Train shape      : (48000, 14)
Validation shape : (12000, 13)
Template shape   : (12000, 2)
__________________________________________________
Train date range : 2025-01-01 → 2025-10-31
Val date range   : 2025-11-01 → 2025-12-31
__________________________________________________
December Shape: (31, 7)
    Dates: 2025-12-01 -> 2025-12-31
    Columns: Index(['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date',
       'predicted_rate'],
      dtype='str')
__________________________________________________


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


## Inspect Data

In [43]:
print(f"Data metadata:")
df.info()

Data metadata:
<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  str    
 1   pickup        48000 non-null  str    
 2   delivery      48000 non-null  str    
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  str    
 9   weight        47700 non-null  float64
 10  date          48000 non-null  str    
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), str(5)
memory usage: 5.1 MB


In [44]:
# Percentage of Missing Values
df.isna().sum() / len(df)

load_id         0.000000
pickup          0.000000
delivery        0.000000
pickup_lat      0.000000
pickup_lon      0.000000
delivery_lat    0.000000
delivery_lon    0.000000
distance        0.000000
equipment       0.000000
weight          0.006250
date            0.000000
market_index    0.007792
quote_signal    0.000000
posted_rate     0.000000
dtype: float64

In [45]:
df.duplicated().sum()

np.int64(0)

In [46]:
print("=== Missing Values % ===")
display(pd.DataFrame({
    "train": df.isna().mean().round(4),
    "validation": val_df.isna().mean().round(4)
}))

=== Missing Values % ===


,train,validation
date,0.0000,0.0000
delivery,0.0000,0.0000
delivery_lat,0.0000,0.0000
delivery_lon,0.0000,0.0000
distance,0.0000,0.0000
equipment,0.0000,0.0000
load_id,0.0000,0.0000
market_index,0.0078,0.0208
pickup,0.0000,0.0000
pickup_lat,0.0000,0.0000


In [47]:
print("\nNegative weight  → train:", (df["weight"] < 0).sum(),
      "| validation:", (val_df["weight"] < 0).sum())
print("Duplicates in train:", df.duplicated().sum())


Negative weight  → train: 292 | validation: 145
Duplicates in train: 0


## Clean Data
- Missing Values -> Drop
    - As There are only 2 columns and each has small percentage of missings
- Duplicates -> Not Found
- Fix Data Types
    - Handle dates

In [48]:
# Reusable Preprocessing Function
def preprocess(data: pd.DataFrame, le:LabelEncoder=None, weight_median:pd.Series=None, mi_global:float=None, is_train:bool=True):
    """
    Clean + feature engineering.
    When is_train=True  → fit encoders / compute statistics
    When is_train=False → use the fitted objects
    """
    df = data.copy()
    
    # 1. Basic Cleaning
    
    #   Remove Spaces
    for col in ["pickup", "delivery", "equipment"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    
    #   Convert datetime
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # 2. Negative & Missing Weight
    df['weight'] = df['weight'].abs()   # # fix negative weights
    
    if is_train:
        weight_median = df.groupby("equipment")['weight'].median()
        mi_global = df['market_index'].median()
    
    df['weight'] = (
        df['weight']
        .fillna(df['equipment'].map(weight_median))
        .fillna(weight_median.median())
    )
    
    # 3. Market Index Imputation
    if is_train:
        daily = df.groupby(df['date'].dt.date)['market_index'].median()
        daily.index = pd.to_datetime(daily.index)
        full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
        daily = daily.reindex(full_idx).interpolate(limit_direction='both')
        df['market_index'] = df['market_index'].fillna(df['date'].dt.normalize().map(daily))
    
    df['market_index'] = df['market_index'].fillna(mi_global)
    
    # 4. Equipment Encoding
    if is_train:
        le = LabelEncoder()
        df['equipment'] = le.fit_transform(df['equipment'])
    else:
        # Handle unseen categories safely
        known = set(le.classes_)
        df['equipment'] = df['equipment'].apply(lambda x: x if x in known else le.classes_[0])
        df['equipment'] = le.transform(df['equipment'])
    
    # 5. Date Features
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek # 0 = Mon..., 6=Sun
    df['day'] = df['date'].dt.day
    
    # 6. Feature Engineering
    df['lat_diff'] = df['delivery_lat'] - df['pickup_lat']
    df['lon_diff'] = df['delivery_lon'] - df['pickup_lon']
    df['dist_x_mi'] = df['distance'] * df['market_index']
    
    return df, le, weight_median, mi_global

In [49]:
# Apply Preprocessing on Train
df_clean, le, weight_median, mi_global = preprocess(df, is_train=True)

print("After cleaning – missing values:")
print(df_clean.isna().sum())
print("\nShape:", df_clean.shape)
df_clean.head()

After cleaning – missing values:
load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
posted_rate     0
month           0
day_of_week     0
day             0
lat_diff        0
lon_diff        0
dist_x_mi       0
dtype: int64

Shape: (48000, 20)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate,month,day_of_week,day,lat_diff,lon_diff,dist_x_mi
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,0,30658.0,2025-01-01,0.95684,2.39595,645.41,1,2,1,0.07786,4.04342,262.461212
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,2,17555.0,2025-01-01,0.97623,2.43355,679.97,1,2,1,1.13195,3.82196,273.832515
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,0,31721.0,2025-01-01,1.00971,1.84491,1802.54,1,2,1,5.07979,-14.56161,977.197338
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,0,32333.0,2025-01-01,0.94518,1.87712,1827.28,1,2,1,-4.70395,-14.10889,912.476772
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,2,35183.0,2025-01-01,0.98480,2.56300,1380.28,1,2,1,3.46454,6.29428,533.663120


## Feature Selection

In [50]:
FEATURES = [
    "pickup_lat", "pickup_lon",
    "delivery_lat", "delivery_lon",
    "distance", "weight",
    "market_index", "quote_signal",
    "equipment", "month", "day_of_week",
    "lat_diff", "lon_diff", "dist_x_mi"
]

TARGET = 'posted_rate'

In [51]:
X = df_clean[FEATURES]
y = df_clean[TARGET].values

print("Number of features:", X.shape[1])
X.head()

Number of features: 14


,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,equipment,month,day_of_week,lat_diff,lon_diff,dist_x_mi
0,38.09122,-76.78906,38.16908,-72.74564,274.3,30658.0,0.95684,2.39595,0,1,2,0.07786,4.04342,262.461212
1,38.09122,-76.78906,39.22317,-72.96710,280.5,17555.0,0.97623,2.43355,2,1,2,1.13195,3.82196,273.832515
2,39.22317,-72.96710,44.30296,-87.52871,967.8,31721.0,1.00971,1.84491,0,1,2,5.07979,-14.56161,977.197338
3,39.55328,-72.18051,34.84933,-86.28940,965.4,32333.0,0.94518,1.87712,0,1,2,-4.70395,-14.10889,912.476772
4,31.83025,-94.38343,35.29479,-88.08915,541.9,35183.0,0.98480,2.56300,2,1,2,3.46454,6.29428,533.663120


## Split Data
- Training -> Modeling
- Testing ->  Evaluation

In [52]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.25,
#     random_state=42
# )

# print(f"Training Data Shape = {X_train.shape}")
# print(f"Testing Data Shape = {X_test.shape}")

In [53]:
# print(f"Training Data Size = {(X_train.shape[0] / len(df))*100:.2f}%")
# print(f"Testing Data Size = {(X_test.shape[0] / len(df))*100:.2f}%")

In [54]:
# Time-based Split
train_mask = df_clean['month'].between(1, 8)
test_mask = df_clean['month'].between(9, 10)

X_train = df_clean.loc[train_mask, FEATURES]
y_train = df_clean.loc[train_mask, 'posted_rate'].values

X_test = df_clean.loc[test_mask, FEATURES]
y_test = df_clean.loc[test_mask, 'posted_rate'].values

In [55]:
print(f"Train months 1-8 : {X_train.shape}")
print(f"Test  months 9-10: {X_test.shape}")
print(f"Train ratio: {len(X_train) / len(df_clean):.1%}")
print(f"Test  ratio: {len(X_test)  / len(df_clean):.1%}")

Train months 1-8 : (38477, 14)
Test  months 9-10: (9523, 14)
Train ratio: 80.2%
Test  ratio: 19.8%


## Machine Learning Modeling

In [56]:
model_portfolio = {
    'Linear Regression': LinearRegression(n_jobs=-1),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        n_jobs=-1,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2,
        subsample=0.8,
        random_state=42
    ),
    'Support Vector': LinearSVR(
        C=1.0,
        epsilon=0.1,
        max_iter=5000,
        dual=True,
        random_state=42
    )
}

In [57]:
print("Start Models Training.....")

results = {}
for model_name, model in model_portfolio.items():
    print(f"    {model_name} Training....")
    t0 = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - t0
    print(f"    Training Time = {training_time:.2f} seconds")
    y_pred = model.predict(X_test)
    y_pred_train = model.predict(X_train)
    
    results[model_name] = {
        'training_time': training_time,
        'train_r2': model.score(X_train, y_train),
        'test_r2': model.score(X_test, y_test),
        'train_mae': mae(y_train, y_pred_train),
        'test_mae': mae(y_test, y_pred),
        'train_mse': mse(y_train, y_pred_train),
        'test_mse': mse(y_test, y_pred)
    }
    print("Model Trained Successfully")
    print(f"    time={training_time:.1f}s | test_r2={results[model_name]['test_r2']} | test_mae={results[model_name]['test_mae']}")
    print("_"*50)

Start Models Training.....
    Linear Regression Training....
    Training Time = 0.03 seconds
Model Trained Successfully
    time=0.0s | test_r2=0.8229202060534524 | test_mae=149.65840244911604
__________________________________________________
    Random Forest Training....
    Training Time = 1.84 seconds
Model Trained Successfully
    time=1.8s | test_r2=0.8222579936607619 | test_mae=150.9134083275389
__________________________________________________
    Gradient Boosting Training....
    Training Time = 17.93 seconds
Model Trained Successfully
    time=17.9s | test_r2=0.8248293615342327 | test_mae=132.38488210182078
__________________________________________________
    Support Vector Training....
    Training Time = 46.23 seconds
Model Trained Successfully
    time=46.2s | test_r2=0.8173826395192736 | test_mae=176.5315231414201
__________________________________________________


In [58]:
print("Models Comparison")

results_df = pd.DataFrame(results).T
results_df = results_df.round(4).sort_values('test_r2', ascending=False)

results_df

Models Comparison


,training_time,train_r2,test_r2,train_mae,test_mae,train_mse,test_mse
Gradient Boosting,17.9334,0.8743,0.8248,105.8601,132.3849,274122.0196,407943.6613
Linear Regression,0.0303,0.8411,0.8229,132.6668,149.6584,346464.9389,412389.7711
Random Forest,1.8362,0.9108,0.8223,102.3533,150.9134,194358.4593,413931.9550
Support Vector,46.2261,0.8349,0.8174,163.8256,176.5315,359956.6600,425285.8545


## Hyperparameter Tuning

In [59]:
param_grids = {
    "Linear Regression": {
        "fit_intercept": [True, False]
    },
    "Random Forest": {
        "n_estimators": [150, 250, 350],
        "max_depth": [10, 14, 18, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    "Gradient Boosting": {
        "n_estimators": [150, 250, 350],
        "learning_rate": [0.03, 0.05, 0.08],
        "max_depth": [3, 4, 5],
        "min_samples_split": [2, 5],
        "subsample": [0.7, 0.8, 0.9]
    }
}

In [60]:
best_models = {}
print("Start Hyperparameter Tuning...\n")

for name, model in model_portfolio.items():
    if name == 'Support Vector':    # skip (slow + weak)
        continue
    
    print(f"Tuning → {name}")
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids[name],
        n_iter=20,
        scoring='neg_mean_squared_error',
        cv=5,
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"Best CV MSE : {-search.best_score_:.1f}")
    print(f"Best Params : {search.best_params_}")
    print("-" * 60)

Start Hyperparameter Tuning...

Tuning → Linear Regression
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best CV MSE : 348004.5
Best Params : {'fit_intercept': False}
------------------------------------------------------------
Tuning → Random Forest
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV MSE : 346366.4
Best Params : {'n_estimators': 350, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}
------------------------------------------------------------
Tuning → Gradient Boosting
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV MSE : 344834.9
Best Params : {'subsample': 0.8, 'n_estimators': 150, 'min_samples_split': 2, 'max_depth': 3, 'learning_rate': 0.05}
------------------------------------------------------------


In [61]:
# Evaluate tuned models on hold-out test
print("\n=== Tuned Models on Hold-out Test ===")
tuned_results = {}
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    tuned_results[name] = {
        "test_r2":  round(r2_score(y_test, y_pred), 4),
        "test_mae": round(mae(y_test, y_pred), 2),
        "test_mse": round(mse(y_test, y_pred), 1)
    }
display(pd.DataFrame(tuned_results).T.sort_values("test_r2", ascending=False))


=== Tuned Models on Hold-out Test ===


,test_r2,test_mae,test_mse
Gradient Boosting,0.8248,131.26,408034.1
Random Forest,0.8239,134.37,410007.1
Linear Regression,0.8233,146.96,411489.8


## Save the Best Model
- Select & Save Best Model + Artifacts

In [62]:
best_name = max(tuned_results, key=lambda k: tuned_results[k]["test_r2"])
best_model = best_models[best_name]

print(f"Selected best model: {best_name}")
print(f"Test R²  = {tuned_results[best_name]['test_r2']}")
print(f"Test MAE = {tuned_results[best_name]['test_mae']}")

Selected best model: Gradient Boosting
Test R²  = 0.8248
Test MAE = 131.26


In [63]:
# Save everything needed for inference
joblib.dump({
    "model": best_model,
    "label_encoder": le,
    "weight_median": weight_median,
    "mi_global": mi_global,
    "features": FEATURES
}, "artifacts.joblib")

['artifacts.joblib']

## Inference
- **Inference on Validation Set -> validation_predictions.csv**

In [64]:
print("Preprocessing validation data...")
val_clean, _, _, _ = preprocess(
    val_df,
    le=le,
    weight_median=weight_median,
    mi_global=mi_global,
    is_train=False
)

X_val = val_clean[FEATURES]
print("Validation features shape:", X_val.shape)

Preprocessing validation data...
Validation features shape: (12000, 14)


In [65]:
print("Generating predictions...")
val_preds = best_model.predict(X_val)

# Build submission (safe merge on load_id)
submission = template[["load_id"]].copy()
pred_df = pd.DataFrame({
    "load_id": val_clean["load_id"].values,
    "predicted_rate": val_preds
})
submission = submission.merge(pred_df, on="load_id", how="left")


# Final checks
assert submission.shape[0] == 12000, "Wrong number of rows!"
assert submission["predicted_rate"].isna(
).sum() == 0, "There are missing predictions!"

submission.to_csv("../validation_predictions.csv", index=False)
print("\n✅ validation_predictions.csv saved successfully!")
print(submission.head(10))
print("\nShape:", submission.shape)
print("Predicted rate stats:")
print(submission["predicted_rate"].describe().round(2))

Generating predictions...

✅ validation_predictions.csv saved successfully!
     load_id  predicted_rate
0  TE-000001      819.286289
1  TE-000002     4873.586832
2  TE-000003     5231.687221
3  TE-000004     4014.950807
4  TE-000005     1777.224857
5  TE-000006     4309.909852
6  TE-000007     5158.708813
7  TE-000008     2800.735657
8  TE-000009     1096.752977
9  TE-000010     1153.769280

Shape: (12000, 2)
Predicted rate stats:
count    12000.00
mean      2324.99
std       1324.73
min        276.09
25%       1254.39
50%       2009.18
75%       3272.34
max       7763.13
Name: predicted_rate, dtype: float64


- **December Predictions (Fixed Route - Lexington → Fort Wayne)**


In [70]:
# 1. Load artifacts
artifacts = joblib.load("artifacts.joblib")
model = artifacts["model"]
le = artifacts["label_encoder"]
WEIGHT_MEDIAN = artifacts["weight_median"]
MI_GLOBAL = artifacts["mi_global"]
FEATURES = artifacts["features"]

# 2. Load December file
dec = pd.read_csv("../data/december_chart_inputs.csv")
print("Original December shape:", dec.shape)

# 3. Get coordinates from training data
train = pd.read_csv("../data/train_test.csv")

lexington_coords = train[train["pickup"] =="Lexington"][["pickup_lat", "pickup_lon"]].median()
fortwayne_coords = train[train["delivery"] == "Fort Wayne"][["delivery_lat", "delivery_lon"]].median()

# 4. Build complete December dataframe
dec_full = dec.copy()

dec_full["pickup_lat"] = lexington_coords["pickup_lat"]
dec_full["pickup_lon"] = lexington_coords["pickup_lon"]
dec_full["delivery_lat"] = fortwayne_coords["delivery_lat"]
dec_full["delivery_lon"] = fortwayne_coords["delivery_lon"]

# ---- market_index: stable with small random fluctuation ----
np.random.seed(42)
base_mi = MI_GLOBAL
noise = np.random.normal(
    loc=0, scale=0.025, size=len(dec_full))
dec_full["market_index"] = base_mi * (1 + noise)

# ---- quote_signal ----
base_qs = train["quote_signal"].median()
noise_qs = np.random.normal(loc=0, scale=0.02, size=len(dec_full))
dec_full["quote_signal"] = base_qs * (1 + noise_qs)

print("market_index range:", dec_full["market_index"].min().round(
    4), "→", dec_full["market_index"].max().round(4))
print("quote_signal range:", dec_full["quote_signal"].min().round(
    4), "→", dec_full["quote_signal"].max().round(4))

# 5. Preprocess
dec_clean, _, _, _ = preprocess(
    dec_full,
    le=le,
    weight_median=WEIGHT_MEDIAN,
    mi_global=MI_GLOBAL,
    is_train=False
)

# 6. Predict
X_dec = dec_clean[FEATURES]
preds = model.predict(X_dec)

dec["predicted_rate"] = preds
dec.to_csv("../december_predictions.csv", index=False)

print("\n✅ december_predictions.csv saved!")
print(dec[["date", "predicted_rate"]].head(10))
print("\nStats:")
print(dec["predicted_rate"].describe().round(2))

Original December shape: (31, 7)
market_index range: 1.0053 → 1.0975
quote_signal range: 1.9752 → 2.1319

✅ december_predictions.csv saved!
         date  predicted_rate
0  2025-12-01      839.464899
1  2025-12-02      842.854015
2  2025-12-03      839.464899
3  2025-12-04      839.464899
4  2025-12-05      842.854015
5  2025-12-06      842.854015
6  2025-12-07      839.464899
7  2025-12-08      839.464899
8  2025-12-09      842.854015
9  2025-12-10      839.464899

Stats:
count     31.00
mean     841.82
std        1.43
min      839.46
25%      841.17
50%      842.85
75%      842.85
max      842.85
Name: predicted_rate, dtype: float64


In [71]:
# Quick sanity check – reload artifacts
artifacts = joblib.load("artifacts.joblib")
print("Loaded model type:", type(artifacts["model"]))
print("Features used    :", artifacts["features"])
print("Test score again :", artifacts["model"].score(X_test, y_test))

Loaded model type: <class 'sklearn.ensemble._gb.GradientBoostingRegressor'>
Features used    : ['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'market_index', 'quote_signal', 'equipment', 'month', 'day_of_week', 'lat_diff', 'lon_diff', 'dist_x_mi']
Test score again : 0.8247905319444782
